# autoresearch-xla: Autonomous LLM Research on Colab & Kaggle (GPU + TPU)

A pure-**JAX/XLA** port of [Andrej Karpathy's `autoresearch`](https://github.com/karpathy/autoresearch) designed to run seamlessly on **both Google Colab and Kaggle** across **both GPUs and TPUs**:

- **GPUs**: Colab T4 / L4 / A100 / H100, Kaggle T4 / 2×T4 / P100 (auto `bf16` on Ampere+ or `fp16` + dynamic loss scaling on T4/P100)
- **TPUs**: Colab TPU `v2-8` / `v5e-1` / `v6e-1`, Kaggle TPU `v3-8` / `v5e-8` (auto `bf16` + automatic multi-chip SPMD data parallelism across all 8 chips!)
- **TPU-Safe Architecture**: The notebook kernel never imports JAX directly; every experiment runs in an isolated subprocess so Cloud TPU `libtpu_lockfile` conflicts never happen and HBM/VRAM is completely freed between runs.
- **Persistent & Resume-Safe**: Syncs `results.tsv`, `experiments.jsonl`, `best_train.py`, and `progress.png` to Google Drive (Colab) or `/kaggle/working` (Kaggle) after every single experiment.

### Quickstart
1. Select your runtime accelerator (**GPU** or **TPU**) in Colab (`Runtime -> Change runtime type`) or Kaggle (`Settings -> Accelerator`).
2. Add your LLM API key to **Colab Secrets** (🔑 sidebar) or **Kaggle Secrets** (`Add-ons -> Secrets`): `GEMINI_API_KEY`, `ANTHROPIC_API_KEY`, `OPENAI_API_KEY`, or `OPENROUTER_API_KEY`.
3. Run all cells top to bottom and wake up to a lower `val_bpb`!


## 1. Environment Setup (Colab / Kaggle Auto-Detection)


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

# Detect execution environment
IN_COLAB = "google.colab" in sys.modules or os.path.exists("/content")
IN_KAGGLE = os.path.exists("/kaggle/working")
ENV_NAME = "colab" if IN_COLAB else ("kaggle" if IN_KAGGLE else "local")
print(f"Detected environment: {ENV_NAME}")

# Optional: Set your GitHub fork URL if launching this notebook standalone without local files
REPO_URL = ""  # e.g. "https://github.com/<your-user>/autoresearch-xla.git"
MOUNT_GOOGLE_DRIVE = True  # Persist progress to Google Drive on Colab

SYNC_DIR = None
if IN_COLAB and MOUNT_GOOGLE_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        SYNC_DIR = "/content/drive/MyDrive/autoresearch-xla-runs"
    except Exception as exc:
        print(f"Skipping Google Drive mount: {exc}")
        SYNC_DIR = "/content/autoresearch-runs"
elif IN_KAGGLE:
    SYNC_DIR = "/kaggle/working/autoresearch-runs"
else:
    SYNC_DIR = str(Path.cwd() / "saved_runs")

# Locate or clone repo
if Path("train.py").exists() and Path("prepare.py").exists():
    REPO_DIR = Path.cwd()
elif REPO_URL:
    target = Path("/content/autoresearch-xla" if IN_COLAB else ("/kaggle/working/autoresearch-xla" if IN_KAGGLE else "./autoresearch-xla"))
    if not target.exists():
        subprocess.run(["git", "clone", REPO_URL, str(target)], check=True)
    os.chdir(target)
    REPO_DIR = target
else:
    raise FileNotFoundError(
        "train.py / prepare.py not found in current directory. "
        "Either upload the repo files or set REPO_URL above."
    )

print(f"Working repo: {REPO_DIR}")
print(f"Sync dir:     {SYNC_DIR}")

# Install lightweight non-JAX dependencies ONLY — NEVER overwrite preinstalled Colab/Kaggle JAX/libtpu!
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "tiktoken", "pyarrow", "requests", "tokenizers", "matplotlib"],
    check=True,
)
# Best-effort rustbpe install (prepare.py seamlessly falls back to HF `tokenizers` if wheel unavailable)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "rustbpe"], check=False)

# Verify JAX is present (only install CPU fallback if running on a bare local machine without JAX)
jax_check = subprocess.run([sys.executable, "-c", "import jax; print('JAX', jax.__version__)"], capture_output=True, text=True)
if jax_check.returncode == 0:
    print(jax_check.stdout.strip())
else:
    print("JAX not detected; installing jax...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "jax"], check=True)

## 2. Detect GPU / TPU Hardware (Subprocess-Isolated)

> **Important TPU note:** We probe JAX inside a short-lived subprocess so the Jupyter kernel itself **never** grabs the exclusive Cloud TPU device lock (`/tmp/libtpu_lockfile`). Every `train.py` run gets clean, exclusive access to all GPU/TPU chips.


In [ ]:
import subprocess, sys

probe = subprocess.run(
    [
        sys.executable,
        "-c",
        (
            "from prepare import detect_hardware; h = detect_hardware(); "
            "print(f'Accelerator:      {h.device_kind} x{h.num_devices} ({h.platform})\n'"
            "      f'HBM / VRAM:       {h.hbm_per_device_gb:.1f} GB per device\n'"
            "      f'Compute dtype:    {h.compute_dtype}\n'"
            "      f'Est. peak TFLOPS: {h.peak_tflops_total:.1f}\n'"
            "      f'Auto profile:     depth={h.default_depth}, total_batch={h.default_total_batch_size:,}, '"
            "      f'per_dev_batch={h.default_per_device_batch}, window={h.default_window_pattern}, remat={h.default_remat}')"
        ),
    ],
    capture_output=True,
    text=True,
)
print(probe.stdout)
if probe.returncode != 0:
    print(probe.stderr)
    raise RuntimeError("Hardware probe failed.")

## 3. Prepare Dataset & BPE Tokenizer (One-Time Setup)

- `climbmix` *(default)*: Karpathy's `climbmix-400b-shuffle` shards + pinned `shard_06542.parquet` validation shard (~100 MB/shard).
- `tinystories`: Smaller & faster convergence — great for quick demos on a free Colab T4 GPU.
- `synthetic`: Zero-network offline corpus for instant smoke testing.


In [ ]:
import subprocess, sys

DATASET = "climbmix"  # "climbmix" | "tinystories" | "synthetic"
NUM_SHARDS = 4        # 4 shards (~400MB) is fast on Colab/Kaggle; use 10 for longer TPU runs

subprocess.run(
    [sys.executable, "prepare.py", "--dataset", DATASET, "--num-shards", str(NUM_SHARDS)],
    check=True,
)

## 4. (Optional) Standalone Baseline Run

Run `train.py` directly once to watch XLA compilation, step speed, MFU%, and your baseline `val_bpb` live. Skip this cell if you want Cell 5 to run and record the baseline automatically.


In [ ]:
# Uncomment to run a standalone 5-minute baseline test directly:
# !python3 train.py
print("Ready! Uncomment `!python3 train.py` to test standalone, or proceed to Cell 5 for the autonomous loop.")

## 5. Launch the Autonomous Research Agent Loop

This runs Karpathy's propose → commit → train (5 min) → evaluate `val_bpb` → keep/revert ratchet loop continuously!
It reads your API key automatically from **Colab Secrets**, **Kaggle Secrets**, or environment variables (`GEMINI_API_KEY`, `ANTHROPIC_API_KEY`, `OPENAI_API_KEY`, `OPENROUTER_API_KEY`).


In [ ]:
import datetime as dt
from pathlib import Path
import agent

# ---- User Configuration ----
PROVIDER = "auto"         # "auto" | "gemini" | "anthropic" | "openai" | "openrouter" | "openai-compat"
MODEL = None              # None -> smart default (gemini-2.5-pro, claude-sonnet-4-6, gpt-4.1, ...)
API_KEY = None            # Optional override; leave None to use Colab/Kaggle Secrets or env vars
BASE_URL = None           # Only needed for provider="openai-compat" (e.g. Ollama / vLLM / Groq)

RUN_TAG = f"{ENV_NAME}-{dt.datetime.now().strftime('%b%d').lower()}"
MAX_EXPERIMENTS = 50      # ~12 experiments/hour at 5 min each
MAX_HOURS = 8.5           # Cleanly stops before Kaggle's 9h TPU / Colab session cutoff
FIX_CRASHES = True        # Give the LLM 1 automated retry to fix simple typos/shape bugs
RESUME = True             # Automatically resume from SYNC_DIR if session restarted
# ----------------------------

run_sync_dir = Path(SYNC_DIR) / RUN_TAG if SYNC_DIR else None

provider, keys, model, base_url = agent.resolve_provider_and_keys(
    PROVIDER, api_key=API_KEY, model=MODEL, base_url=BASE_URL
)
print(f"Using provider={provider}, model={model}, api_keys={len(keys)}")

client = agent.LLMClient(
    provider=provider,
    api_keys=keys,
    model=model,
    base_url=base_url,
)

rows = agent.run_autoresearch_loop(
    repo_dir=Path.cwd(),
    tag=RUN_TAG,
    llm_complete=client.complete,
    max_experiments=MAX_EXPERIMENTS,
    max_hours=MAX_HOURS,
    sync_dir=run_sync_dir,
    resume=RESUME,
    fix_crashes=FIX_CRASHES,
)

## 6. Inspect Progress Chart, Leaderboard & Winning Diff


In [ ]:
import subprocess
from pathlib import Path
from IPython.display import Image, display
import agent

tsv_path = Path("results.tsv")
png_path = Path("progress.png")

if tsv_path.exists():
    agent.plot_progress(tsv_path, png_path)
    if png_path.exists():
        display(Image(filename=str(png_path)))

    rows = agent.load_results_tsv(tsv_path)
    kept = [r for r in rows if r["status"] == "keep"]
    print(f"Total experiments: {len(rows)} | Kept: {len(kept)} | Crashes: {sum(r['status']=='crash' for r in rows)}")
    print("\nWinning lineage:")
    for r in kept:
        print(f"  {r['commit']}  val_bpb={r['val_bpb']:.6f}  mem={r['memory_gb']:4.1f}GB  {r['description']}")

    print("\nRecent git commits:")
    print(subprocess.run(["git", "log", "--oneline", "-n", "15"], capture_output=True, text=True).stdout)
else:
    print("No results.tsv yet — run Cell 5 first!")